In [7]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = "1"

import re
import csv
import sys
import time
import json
import pickle
import pygame
import numpy as np

from pymavlink import mavutil
from collections import deque

# Пути и константы
CLASS_MODEL_PATH = "DATA/real_time/lgb_final_model.pkl"     # Модель классификации повреждений
REG_MODEL_PATH = "DATA/real_time/ordinal_pipeline.pkl"      # Модель регрессии для оценки
FEATURES_PATH = "DATA/real_time/selected_50_features.json"  # Список отобранных признаков
FOLDER_NAME = "DATA/test_realtime_system"                   # Папка для сохранения данных

WIDTH, HEIGHT = 1000, 600                                   # Размеры окна приложения

# Описательное название для каждого класса повреждений (0-5)
CLASS_LABELS = { 
    0: "НОРМА (Винты целы)", 
    1: "ЛЕГКОЕ ПОВРЕЖДЕНИЕ: 1 винт (0.5 см)",
    2: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 1 винт (1.0 см)",
    3: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 1 винт (1.5 см)",
    4: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 2 винта (0.5 см)",
    5: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 3 винта (0.5 см)"
}

# RGB цвета для визуализации разных классов (зеленый-желтый-оранжевый-красный)
CLASS_COLORS = [(0, 255, 100), (255, 255, 0), (255, 170, 0), 
                (255, 0, 50), (255, 170, 0), (255, 0, 50)]

# Словарь функций для вычисления статистических характеристик сигналов
STATS_MAP = {
    "Min": np.min, 
    "Max": np.max, 
    "Mean": np.mean, 
    "Ptp": np.ptp, 
    "L1": lambda x: np.sum(np.abs(x)), 
    "CumsumFinal": np.sum
}

# Загрузка моделей
try:
    with open(CLASS_MODEL_PATH, "rb") as f:  # Загрузка модели классификации
        clf_model = pickle.load(f)
        
    with open(REG_MODEL_PATH, "rb") as f:    # Загрузка пайплайна регрессии 
        reg_pipeline = pickle.load(f)
        
    with open(FEATURES_PATH) as f:           # Загрузка списка отобранных признаков
        selected_features = json.load(f)
        
    print("Модели и конфигурация признаков успешно загружены.")
    
except Exception as e:
    print(f"Ошибка загрузки: {e}")
    sys.exit()

# Распаковка моделей
reg_models = reg_pipeline["models"]
scaler = reg_pipeline["scaler"]
l_min, l_max = reg_pipeline["latent_min"], reg_pipeline["latent_max"]

# Подготовка признаков
print(selected_features)
feature_list = []
for feat in selected_features:
    prefix, stat = feat.rsplit("_", 1)
    feature_list.append((prefix + "_", stat))
print(selected_features)

def extract_ordered_features(window):
    """
    Извлечение признаков из окна данных телеметрии
    
    Параметры:
        - window - numpy массив размером (n_samples, 6) с данными 
          гироскопа и акселерометра (столбцы: gx, gy, gz, ax, ay, az)
    
    Возвращает:
        - numpy массив признаков для моделей машинного обучения
    """
    names = ['gx', 'gy', 'gz', 'ax', 'ay', 'az']
    sigs = {}
    for i, name in enumerate(names):
        raw = window[:, i]
        d1 = np.diff(raw)
        sigs[f"{name}_TD_"] = raw
        sigs[f"{name}_TD_dif_"] = d1
        sigs[f"{name}_TD_dif_dif_"] = np.diff(d1)
        
    return np.array([STATS_MAP[stat](sigs[prefix]) for prefix, stat in feature_list], dtype=np.float32)


def get_file_path():
    """
    Генерация уникального пути для сохранения файла с данными
    
    Создает папку если её нет, определяет следующий номер файла
    на основе существующих файлов в папке
    
    Возвращает:
        - строку с путем к новому файлу
    """
    os.makedirs(FOLDER_NAME, exist_ok=True)
    nums = [int(m.group(1)) for f in os.listdir(FOLDER_NAME) if (m := re.search(r'data_(\d+)\.csv', f))]
    next_num = max(nums) + 1 if nums else 1
    return os.path.join(FOLDER_NAME, f'pixhawk_imu_data_{next_num}.csv')


Модели и конфигурация признаков успешно загружены.
['gy_TD_dif_dif_Min', 'gz_TD_dif_Min', 'ax_TD_Min', 'gz_TD_dif_dif_Min', 'ax_TD_dif_dif_Min', 'ay_TD_Min', 'ax_TD_dif_Min', 'gy_TD_Min', 'az_TD_Min', 'gy_TD_dif_Min', 'gx_TD_dif_Min', 'ay_TD_dif_Min', 'az_TD_dif_dif_Max', 'ay_TD_dif_dif_Max', 'ax_TD_dif_dif_Max', 'gz_TD_dif_dif_Max', 'gz_TD_dif_Max', 'gz_TD_Max', 'ax_TD_Max', 'ay_TD_Max', 'gy_TD_dif_dif_Max', 'az_TD_Max', 'ax_TD_dif_Max', 'gy_TD_dif_Max', 'gx_TD_dif_Max', 'gy_TD_Max', 'az_TD_dif_Max', 'gx_TD_Min', 'gy_TD_dif_dif_CumsumFinal', 'gz_TD_L1', 'az_TD_L1', 'gz_TD_dif_dif_L1', 'gx_TD_dif_L1', 'gy_TD_dif_dif_L1', 'ax_TD_dif_dif_L1', 'ax_TD_L1', 'ay_TD_dif_L1', 'ay_TD_L1', 'gz_TD_dif_L1', 'az_TD_dif_dif_L1', 'ay_TD_dif_dif_L1', 'az_TD_dif_Mean', 'gy_TD_Mean', 'gx_TD_Ptp', 'gx_TD_dif_Ptp', 'az_TD_dif_dif_Ptp', 'gz_TD_dif_Ptp', 'ay_TD_dif_Ptp', 'ax_TD_dif_dif_Ptp', 'az_TD_dif_Ptp']
['gy_TD_dif_dif_Min', 'gz_TD_dif_Min', 'ax_TD_Min', 'gz_TD_dif_dif_Min', 'ax_TD_dif_dif_Min', 'ay_TD

In [42]:
PORT = 14577      # Порт для подключения к Pixhawk по UDP

WINDOW_SIZE = 32  # Размер окна для анализа (НЕ МЕНЯТЬ - модель обучена под этот размер)
STRIDE = 8        # Шаг скользящего окна

CSV_BATCH_SIZE = 100       # Количество пакетов перед сохранением в CSV
MAX_READ_TIME_SEC = 0.025  # Максимальное время сбора данных за кадр (25 мс)
HISTORY_LEN = 150          # Длина истории для графиков задержки и риска
FPS = 60                   # Частота обновления

# ВАЖНО: Для первого запуска нужны обе строки. Если было нажато "Restart the kernel" 
# после успешного запуска - закомментировать вторую строку. Если программа 
# была остановлена - закомментировать обе строки. Если поменялся порт, то
# в любом случае оставить обе строки.

master = mavutil.mavlink_connection(f'udpin:0.0.0.0:{PORT}')  # Создание UDP-подключения
master.wait_heartbeat()                                       # Ожидание сигнала heartbeat от Pixhawk

# Создание и открытие CSV файла для записи телеметрии
file_path = get_file_path()
csv_file = open(file_path, 'w', newline='')
writer = csv.writer(csv_file)

# Запись заголовков CSV файла (все поля сообщения HIGHRES_IMU и VIBRATION)
writer.writerow([
    'time_usec',              
    'ax', 'ay', 'az', 
    'gx', 'gy', 'gz', 
    'mx', 'my', 'mz', 
    'abs_pressure', 'diff_pressure', 'pressure_alt', 
    'temperature', 'fields_updated', 'id', 
    'vibration_imu_instance', 'vib_x', 'vib_y', 'vib_z', 
    'clipping_0', 'clipping_1', 'clipping_2'
])

# Подготовка структур данных для хранения и обработки потока
imu_buffer = np.zeros((WINDOW_SIZE, 6), dtype=np.float32)  # Буфер для скользящего окна IMU данных 
latency_history = [0.0] * HISTORY_LEN                      # История задержек обработки для графика
risk_history = [0.0] * HISTORY_LEN                         # История значений риска для графика
csv_buf = deque()                                          # Буфер для пакетов перед записью в CSV

# Переменные для отслеживания состояния системы
current_risk = 0                                           # Текущее значение риска (0-100%)
current_class = 0                                          # Текущий класс повреждения
current_latency = 0                                        # Текущая задержка обработки (мс)
packets_received = 0                                       # Всего получено пакетов с начала работы
logged_pkts_total = 0                                      # Всего записано пакетов в файл
new_since_predict = 0                                      # Новых пакетов после последнего прогноза

# Запуск графического интерфейса
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Real-Time Detection System")
clock = pygame.time.Clock()

# Шрифты
f_large = pygame.font.SysFont("verdana", 28, bold=True)
f_med = pygame.font.SysFont("verdana", 22, bold=True)
f_med_small = pygame.font.SysFont("verdana", 20, bold=True)
f_small = pygame.font.SysFont("verdana", 14)

# Бесконечный цикл сбора данных, их обработки и отображения
try:
    running = True
    while running:
        # Проверка действий пользователя (закрытие окна)
        for event in pygame.event.get():
            if event.type == pygame.QUIT: 
                running = False

        # Чтение входящих сообщений в течение заданного времени
        frame_start = time.perf_counter()
        while (time.perf_counter() - frame_start) < MAX_READ_TIME_SEC:
            msg = master.recv_match(blocking=False)
            if not msg: break

            # Получение типа сообщения
            m_type = msg.get_type()

            # Обработка HIGHRES_IMU
            if m_type == 'HIGHRES_IMU':
                csv_buf.append([msg.time_usec, 
                                msg.xacc, msg.yacc, msg.zacc, 
                                msg.xgyro, msg.ygyro, msg.zgyro, 
                                msg.xmag, msg.ymag, msg.zmag, 
                                msg.abs_pressure, msg.diff_pressure, msg.pressure_alt, 
                                msg.temperature, msg.fields_updated, msg.id, 
                                '', '', '', '', '', '', ''])
                
                logged_pkts_total += 1

                # Обновление скользящего окна IMU данных
                imu_buffer = np.roll(imu_buffer, -1, axis=0)
                imu_buffer[-1] = [msg.xgyro, msg.ygyro, msg.zgyro, msg.xacc, msg.yacc, msg.zacc]
                
                packets_received += 1
                new_since_predict += 1

                # Если набралось достаточно новых данных для нового прогноза
                if packets_received >= WINDOW_SIZE and new_since_predict >= STRIDE:
                    t_start = time.perf_counter()                                # Засекаем время начала обработки

                    X_raw = extract_ordered_features(imu_buffer).reshape(1, -1)  # Извлечение признаков
                    current_class = int(clf_model.predict(X_raw)[0])             # Классификация
                    X_scaled = scaler.transform(X_raw)                           # Масштабирование

                    # Суммирование выходов всех бинарных моделей в пайплайне
                    l_scores = [m.decision_function(X_scaled)[0] for m in reg_models]
                    latent_val = np.mean([s[0] if hasattr(s, "__len__") else s for s in l_scores]) 

                    # Нормализация значения риска в проценты (0-100%)
                    denom = (l_max - l_min)
                    current_risk = 0.0 if denom == 0 else np.clip(100 * (latent_val - l_min) / denom, 0, 100)

                    # Замер задержки обработки
                    current_latency = (time.perf_counter() - t_start) * 1000

                    # Обновление истории для графиков
                    risk_history.append(float(current_risk))
                    risk_history.pop(0)
                    
                    latency_history.append(float(current_latency))
                    latency_history.pop(0)
                    
                    # Сбрасываем счетчик новых пакетов
                    new_since_predict = 0 

            # Обработка VIBRATION
            elif m_type == 'VIBRATION':
                csv_buf.append([msg.time_usec, 
                                '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 
                                msg.id, 
                                msg.vibration_x, msg.vibration_y, msg.vibration_z, 
                                msg.clipping_0, msg.clipping_1, msg.clipping_2])
                
                logged_pkts_total += 1

         # Периодическая запись данных в CSV (каждые CSV_BUFFER_SIZE пакетов)
        if len(csv_buf) >= CSV_BATCH_SIZE:
            writer.writerows(csv_buf)
            csv_buf.clear()

        # Обновление графического интерфейса
        screen.fill((15, 15, 25))

        # Определение цвета для текущего класса повреждения
        c_color = CLASS_COLORS[current_class] if 0 <= current_class <= 5 else (200, 200, 200)

        # Панель класса
        pygame.draw.rect(screen, (40, 40, 60), (20, 20, 545, 100), border_radius=10)
        pygame.draw.rect(screen, c_color, (20, 20, 545, 100), width=3, border_radius=10)
        screen.blit(f_small.render("ТЕКУЩИЙ КЛАСС:", True, (220, 220, 220)), (40, 35))
        screen.blit(f_med_small.render(CLASS_LABELS.get(current_class, f"КЛАСС {current_class}"), True, c_color), (40, 65))

        # Панель задержки
        lat_rect = pygame.Rect(WIDTH - 320, 20, 300, 100)
        pygame.draw.rect(screen, (40, 40, 60), lat_rect, border_radius=10)
        screen.blit(f_small.render("ЗАДЕРЖКА", True, (220, 220, 220)), (WIDTH - 300, 40))
        screen.blit(f_med.render(f"{current_latency:.2f} ms", True, (220, 220, 220)), (WIDTH - 300, 65))

        # Мини-график истории задержки
        lat_graph_rect = pygame.Rect(WIDTH - 160, 35, 120, 70)
        pygame.draw.rect(screen, (20, 20, 30), lat_graph_rect)
        max_lat = max(max(latency_history), 5.0)

        # Отрисовка линии графика задержки
        for i in range(1, len(latency_history)):
            x1 = lat_graph_rect.x + int((i - 1) * lat_graph_rect.width / HISTORY_LEN)
            y1 = lat_graph_rect.bottom - int((latency_history[i - 1] / max_lat) * lat_graph_rect.height)
            x2 = lat_graph_rect.x + int(i * lat_graph_rect.width / HISTORY_LEN)
            y2 = lat_graph_rect.bottom - int((latency_history[i] / max_lat) * lat_graph_rect.height)
            pygame.draw.line(screen, (100, 200, 255), (x1, y1), (x2, y2), 2)

        # Информация о логгере (под графиком задержки)
        screen.blit(f_small.render(f"Файл: {os.path.basename(file_path)}", True, (150, 150, 150)), (WIDTH - 318, 128))
        screen.blit(f_small.render(f"Записано пакетов: {logged_pkts_total}", True, (0, 255, 150)), (WIDTH - 318, 152))

        # График риска
        g_rect = pygame.Rect(20, 180, WIDTH - 40, HEIGHT - 220)
        pygame.draw.rect(screen, (20, 20, 30), g_rect, border_radius=5)

        # Горизонтальные линии-маркеры для 10% и 25%
        for p, c in [(10, (0, 200, 100)), (25, (255, 170, 0))]:
            yl = g_rect.bottom - int((p / 100) * g_rect.height)
            pygame.draw.line(screen, c, (g_rect.left, yl), (g_rect.right, yl), 1)
            screen.blit(f_small.render(f"{p}%", True, c), (g_rect.left + 5, yl - 20))

        # Отрисовка линии графика риска
        if len(risk_history) > 1:
            pts = [(g_rect.left + int(i * g_rect.width / (HISTORY_LEN-1)), 
                    g_rect.bottom - int((v / 100) * g_rect.height)) for i, v in enumerate(risk_history)]
    
            # Выбор цвета в зависимости от уровня риска
            s_color = (0, 200, 100) if current_risk < 10 else ((255, 170, 0) if current_risk < 25 else (255, 50, 50))
            pygame.draw.lines(screen, s_color, False, pts, 3)

        # Заголовок графика с текущим значением риска
        screen.blit(f_large.render(f"Вероятность аварии: {current_risk:.1f}%", True, s_color), 
                    (g_rect.left + 20, g_rect.top + 20))

        pygame.display.flip()  # Отображение
        clock.tick(FPS)        # Ограничение FPS

except Exception as e: 
    print(f"Ошибка: {e}")

except KeyboardInterrupt:
    print("Детектирование остановлено пользователем.")

finally:
    if csv_buf: writer.writerows(csv_buf)  # Сохранение оставшихся в буфере данных
    csv_file.close()                      
    pygame.quit()


## Optimized

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = "1"
print(100)
import re
import csv
import sys
import time
import json
import math
import pickle
import pygame
print(100)
import numpy as np
import onnxruntime as ort
from collections import deque
from pymavlink import mavutil

PORT = 14585      # Порт для подключения к Pixhawk по UDP

WINDOW_SIZE = 32  # Размер окна для анализа (НЕ МЕНЯТЬ - модель обучена под этот размер)
STRIDE = 8        # Шаг скользящего окна

CSV_BATCH_SIZE = 100       # Количество пакетов перед сохранением в CSV
MAX_READ_TIME_SEC = 0.025  # Максимальное время сбора данных за кадр (25 мс)
HISTORY_LEN = 150          # Длина истории для графиков задержки и риска
FPS = 60                   # Частота обновления
# ==================== ПУТИ ====================  
REG_MODEL_PATH = "DATA/real_time/hybrid_pipeline_lgb.pkl"      
FEATURES_PATH = "DATA/real_time/selected_50_features.json"  
FOLDER_NAME = "DATA/test_realtime_system"    
ONNX_CLF = "clf_xgb.onnx"
ONNX_REG = "reg_lgb.onnx"

WIDTH, HEIGHT = 1000, 600

CLASS_LABELS = { 
    0: "НОРМА (Винты целы)", 
    1: "ЛЕГКОЕ ПОВРЕЖДЕНИЕ: 1 винт (0.5 см)",
    2: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 1 винт (1.0 см)",
    3: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 1 винт (1.5 см)",
    4: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 2 винта (0.5 см)",
    5: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 3 винта (0.5 см)"
}

CLASS_COLORS = [(0, 255, 100), (255, 255, 0), (255, 170, 0), 
                (255, 0, 50), (255, 170, 0), (255, 0, 50)]

# ==================== NUMBA ====================
try:
    from numba import jit
    NUMBA_AVAILABLE = True
    print("✓ Numba доступна")
except ImportError:
    NUMBA_AVAILABLE = False
    print("✗ Numba не найдена")

# ==================== FEATURE EXTRACTION ====================
if NUMBA_AVAILABLE:
    @jit(nopython=True, cache=True)
    def extract_features_ultra(window, out):
        for axis in range(6):
            x = window[:, axis]
            n = len(x)
            
            # RAW
            min_val = max_val = sum_val = x[0]
            sum_abs = abs(x[0])
            for j in range(1, n):
                val = x[j]
                if val < min_val:
                    min_val = val
                if val > max_val:
                    max_val = val
                sum_val += val
                sum_abs += abs(val)
            
            mean_val = sum_val / n
            ptp_val = max_val - min_val
            
            base = axis * 18
            out[base:base+6] = [min_val, max_val, mean_val, ptp_val, sum_abs, sum_val]
            
            # 1st DIFF
            if n > 1:
                d0 = x[1] - x[0]
                min_val = max_val = sum_val = d0
                sum_abs = abs(d0)
                for j in range(1, n-1):
                    d = x[j+1] - x[j]
                    if d < min_val:
                        min_val = d
                    if d > max_val:
                        max_val = d
                    sum_val += d
                    sum_abs += abs(d)
                out[base+6:base+12] = [min_val, max_val, sum_val/(n-1), max_val-min_val, sum_abs, sum_val]
            else:
                out[base+6:base+12] = [0.0] * 6
            
            # 2nd DIFF
            if n > 2:
                d2_0 = x[2] - 2*x[1] + x[0]
                min_val = max_val = sum_val = d2_0
                sum_abs = abs(d2_0)
                for j in range(2, n-1):
                    d2 = x[j+1] - 2*x[j] + x[j-1]
                    if d2 < min_val:
                        min_val = d2
                    if d2 > max_val:
                        max_val = d2
                    sum_val += d2
                    sum_abs += abs(d2)
                out[base+12:base+18] = [min_val, max_val, sum_val/(n-2), max_val-min_val, sum_abs, sum_val]
            else:
                out[base+12:base+18] = [0.0] * 6
else:
    def extract_features_ultra(window, out):
        for axis in range(6):
            x = window[:, axis]
            base = axis * 18
            out[base:base+6] = [np.min(x), np.max(x), np.mean(x), np.ptp(x), np.sum(np.abs(x)), np.sum(x)]
            if len(x) > 1:
                d = np.diff(x)
                out[base+6:base+12] = [np.min(d), np.max(d), np.mean(d), np.ptp(d), np.sum(np.abs(d)), np.sum(d)]
            if len(x) > 2:
                d2 = np.diff(np.diff(x))
                out[base+12:base+18] = [np.min(d2), np.max(d2), np.mean(d2), np.ptp(d2), np.sum(np.abs(d2)), np.sum(d2)]

# ==================== ЗАГРУЗКА ====================
print("\n🔧 Загрузка компонентов...")

required = [ONNX_CLF, ONNX_REG, REG_MODEL_PATH, FEATURES_PATH]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"✗ Отсутствуют: {missing}")
    sys.exit(1)

try:
    # ONNX Runtime с оптимизациями
    opts = ort.SessionOptions()
    opts.enable_cpu_mem_arena = True
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = 1
    opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    
    sess_clf = ort.InferenceSession(ONNX_CLF, opts, providers=["CPUExecutionProvider"])
    sess_reg = ort.InferenceSession(ONNX_REG, opts, providers=["CPUExecutionProvider"])

    with open(REG_MODEL_PATH, "rb") as f:    
        hybrid = pickle.load(f)
    with open(FEATURES_PATH) as f:           
        selected = json.load(f)
    
    print("✓ Модели загружены")
except Exception as e:
    print(f"✗ {e}")
    sys.exit(1)

# Данные
ordinal_models = hybrid["ordinal_models"]
scaler = hybrid["scaler"]
l_min, l_max = hybrid["latent_min"], hybrid["latent_max"]
r_min, r_max = hybrid["reg_min"], hybrid["reg_max"]
alpha = hybrid["alpha"]

W = np.vstack([m.coef_.ravel() for m in ordinal_models]).astype(np.float32).T
b = np.array([m.intercept_[0] for m in ordinal_models], dtype=np.float32)
mean = scaler.mean_.astype(np.float32)
scale = scaler.scale_.astype(np.float32)
inv_scale = 1.0 / scale

denom_l = l_max - l_min
denom_r = r_max - r_min

print(f"✓ W: {W.shape}, b: {b.shape}")

# Маппинг признаков (108 → 50)
feature_indices = []
for feat in selected:
    prefix, stat = feat.rsplit("_", 1)
    axis = {'gx':0,'gy':1,'gz':2,'ax':3,'ay':4,'az':5}[prefix.split('_')[0]]
    stat_idx = {"Min":0,"Max":1,"Mean":2,"Ptp":3,"L1":4,"CumsumFinal":5}[stat]
    sig_type = 2 if "dif_dif" in prefix else (1 if "dif" in prefix else 0)
    idx = axis * 18 + sig_type * 6 + stat_idx
    feature_indices.append(idx)

feature_indices = np.array(feature_indices, dtype=np.int32)
print(f"✓ {len(feature_indices)} признаков из 108")

print("\n Progrev NUMBA...")
warmup_buffer = np.random.randn(WINDOW_SIZE, 6).astype(np.float32)
warmup_features = np.empty(108, dtype=np.float32)

for i in range(5):
    extract_features_ultra(warmup_buffer, warmup_features)
    print(f"  ������� {i+1}/5", end='\r')

print("\n? JIT ������� ��������\n")

# ==================== CSV ====================
def get_file_path():
    os.makedirs(FOLDER_NAME, exist_ok=True)
    nums = [int(re.search(r'data_(\d+)\.csv', f).group(1)) 
            for f in os.listdir(FOLDER_NAME) if re.search(r'data_(\d+)\.csv', f)]
    next_num = max(nums) + 1 if nums else 1
    return os.path.join(FOLDER_NAME, f'real_imu_data_{next_num}.csv')

# ==================== ПАРАМЕТРЫ ДЛЯ РЕАЛЬНОГО КОПТЕРА ====================
WINDOW_SIZE = 32  # Размер окна для анализа (НЕ МЕНЯТЬ - модель обучена под этот размер)
STRIDE = 8        # Шаг скользящего окна
CSV_BATCH_SIZE = 100       # Количество пакетов перед сохранением в CSV
MAX_READ_TIME_SEC = 0.25  # Максимальное время сбора данных за кадр (25 мс)
HISTORY_LEN = 150          # Длина истории для графиков задержки и риска
FPS = 60                   # Частота обновления

# ==================== ПОДКЛЮЧЕНИЕ К КОПТЕРУ ====================
print(f"\n🔌 Подключение к Pixhawk на порту {PORT}...")


MAVLINK_PORT = '/dev/serial/by-id/usb-3D_Robotics_PX4_FMU_v2.x_0-if00'
BAUDRATE = 921600  

# master = mavutil.mavlink_connection(MAVLINK_PORT, baud=BAUDRATE)
# master.wait_heartbeat()


master = mavutil.mavlink_connection(f'udpin:0.0.0.0:{PORT}')
master.wait_heartbeat()
print("✓ Подключено! Получен heartbeat")

# ==================== ИНИЦИАЛИЗАЦИЯ ====================
file_path = get_file_path()
csv_file = open(file_path, 'w', newline='')
writer = csv.writer(csv_file)

# Запись заголовков CSV файла
writer.writerow([
    'time_usec', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'mx', 'my', 'mz',
    'abs_pressure', 'diff_pressure', 'pressure_alt', 'temperature',
    'fields_updated', 'id', 'vibration_imu_instance', 'vib_x', 'vib_y', 'vib_z'
])

# Предвыделение
imu_buffer = np.zeros((WINDOW_SIZE, 6), dtype=np.float32)
X_raw = np.empty(108, dtype=np.float32)
X_sel = np.empty((1, 50), dtype=np.float32)
X_scaled = np.empty((1, 50), dtype=np.float32)

latency_history = deque([0.0] * HISTORY_LEN, maxlen=HISTORY_LEN)
risk_history = deque([0.0] * HISTORY_LEN, maxlen=HISTORY_LEN)
csv_buf = deque()

current_risk = 0.0
current_class = 0
current_latency = 0.0
packets_received = 0
logged_pkts_total = 0
new_since_predict = 0
total_pred = 0

# Статистика задержек по этапам
total_feature_time = 0.0
total_classifier_time = 0.0
total_scaler_time = 0.0
total_ordinal_time = 0.0
total_regression_time = 0.0
total_fusion_time = 0.0
total_all_time = 0.0

# Pygame
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("REAL-TIME DETECTION - REAL DRONE")
clock = pygame.time.Clock()

font_l = pygame.font.SysFont("verdana", 28, bold=True)
font_m = pygame.font.SysFont("verdana", 22, bold=True)
font_ms = pygame.font.SysFont("verdana", 20, bold=True)
font_s = pygame.font.SysFont("verdana", 14)

print("\n🚀 ЗАПУСК РЕАЛЬНОЙ СИСТЕМЫ\n")
flad = True

# ==================== ОСНОВНОЙ ЦИКЛ ====================
try:
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        
        # Чтение входящих сообщений
        frame_start = time.perf_counter()
        while (time.perf_counter() - frame_start) < MAX_READ_TIME_SEC:
            msg = master.recv_match(blocking=False)
            if not msg: 
                break
            
            m_type = msg.get_type()
            
            # Обработка HIGHRES_IMU
            if m_type == 'HIGHRES_IMU':
                csv_buf.append([msg.time_usec, msg.xacc, msg.yacc, msg.zacc,
                               msg.xgyro, msg.ygyro, msg.zgyro, 
                               msg.xmag, msg.ymag, msg.zmag,
                               msg.abs_pressure, msg.diff_pressure, msg.pressure_alt,
                               msg.temperature, msg.fields_updated, msg.id,
                               '', '', '', ''])
                
                logged_pkts_total += 1
                
                # Обновление скользящего окна
                imu_buffer[:-1] = imu_buffer[1:]
                imu_buffer[-1] = [msg.xgyro, msg.ygyro, msg.zgyro, msg.xacc, msg.yacc, msg.zacc]
                
                packets_received += 1
                new_since_predict += 1
                
                # Прогнозирование
                if packets_received >= WINDOW_SIZE and new_since_predict >= STRIDE:
                    t_start = time.perf_counter()
                    
                    # 1. Features extraction
                    t0 = time.perf_counter()
                    extract_features_ultra(imu_buffer, X_raw)
                    t_feature = time.perf_counter() - t0
                    
                    # 2. Select features
                    t0 = time.perf_counter()
                    X_sel[0, :] = X_raw[feature_indices]
                    t_select = time.perf_counter() - t0
                    
                    # 3. Classifier inference
                    t0 = time.perf_counter()
                    current_class = int(sess_clf.run(None, {"input": X_sel})[0][0])
                    t_classifier = time.perf_counter() - t0
                    
                    # 4. Scaling
                    t0 = time.perf_counter()
                    np.subtract(X_sel, mean, out=X_scaled)
                    np.multiply(X_scaled, inv_scale, out=X_scaled)
                    t_scaler = time.perf_counter() - t0
                    
                    # 5. Ordinal inference
                    t0 = time.perf_counter()
                    ordinal = (np.dot(X_scaled, W) + b).mean()
                    ord_n = (ordinal - l_min) / denom_l
                    t_ordinal = time.perf_counter() - t0
                    
                    # 6. Regression inference
                    t0 = time.perf_counter()
                    reg = sess_reg.run(None, {"input": X_sel})[0][0]
                    reg_n = (reg - r_min) / denom_r
                    t_regression = time.perf_counter() - t0
                    
                    # 7. Fusion
                    t0 = time.perf_counter()
                    current_risk = np.clip(100 * (alpha * ord_n + (1 - alpha) * reg_n), 0, 100)[0]
                    t_fusion = time.perf_counter() - t0
                    
                    t_end = time.perf_counter()
                    
                    # Суммируем задержки
                    total_feature_time += t_feature
                    total_classifier_time += t_classifier
                    total_scaler_time += t_scaler
                    total_ordinal_time += t_ordinal
                    total_regression_time += t_regression
                    total_fusion_time += t_fusion
                    total_all_time += (t_end - t_start)
                    
                    current_latency = (t_end - t_start) * 1000
                    total_pred += 1
                    
                    if flad and total_pred == 10:
                        total_feature_time = 0.0
                        total_classifier_time = 0.0
                        total_scaler_time = 0.0
                        total_ordinal_time = 0.0
                        total_regression_time = 0.0
                        total_fusion_time = 0.0
                        total_all_time = 0.0
                        total_pred = 0
                        flad = False
                    
                    risk_history.append(float(current_risk))
                    latency_history.append(float(current_latency))
                    
                    new_since_predict = 0
            
            # Обработка VIBRATION
            elif m_type == 'VIBRATION':
                csv_buf.append([msg.time_usec, '', '', '', '', '', '', '', '', '', 
                               '', '', '', '', '', msg.id, 
                               msg.vibration_x, msg.vibration_y, msg.vibration_z,
                               msg.clipping_0, msg.clipping_1, msg.clipping_2])
                logged_pkts_total += 1
        
        # CSV batch
        if len(csv_buf) >= CSV_BATCH_SIZE:
            writer.writerows(csv_buf)
            csv_buf.clear()
        
        # ==================== ОТРИСОВКА ====================
        screen.fill((15,15,25))
        color = CLASS_COLORS[current_class] if 0 <= current_class <= 5 else (200,200,200)
        
        # Класс
        pygame.draw.rect(screen, (40,40,60), (20,20,545,100), border_radius=10)
        pygame.draw.rect(screen, color, (20,20,545,100), width=3, border_radius=10)
        screen.blit(font_s.render("ТЕКУЩИЙ КЛАСС:", True, (220,220,220)), (40,35))
        text = CLASS_LABELS.get(current_class, f"КЛАСС {current_class}")
        if len(text) > 35:
            text = text[:32] + "..."
        screen.blit(font_ms.render(text, True, color), (40,65))
        
        # Задержка
        pygame.draw.rect(screen, (40,40,60), (WIDTH-320,20,300,100), border_radius=10)
        screen.blit(font_s.render("ЗАДЕРЖКА", True, (220,220,220)), (WIDTH-300,40))
        screen.blit(font_m.render(f"{current_latency:.3f} ms", True, (220,220,220)), (WIDTH-300,65))
        
        # График задержки
        rect = pygame.Rect(WIDTH-160,35,120,70)
        pygame.draw.rect(screen, (20,20,30), rect)
        max_lat = max(max(latency_history), 5.0)
        lat_list = list(latency_history)
        for i in range(1, len(lat_list)):
            x1 = rect.x + int((i-1)*rect.width/HISTORY_LEN)
            y1 = rect.bottom - int((lat_list[i-1]/max_lat)*rect.height)
            x2 = rect.x + int(i*rect.width/HISTORY_LEN)
            y2 = rect.bottom - int((lat_list[i]/max_lat)*rect.height)
            pygame.draw.line(screen, (100,200,255), (x1,y1), (x2,y2), 2)
        
        # Статус
        screen.blit(font_s.render(f"Файл: {os.path.basename(file_path)}", True, (150,150,150)), (WIDTH-318,128))
        screen.blit(font_s.render(f"Записано: {logged_pkts_total}", True, (0,255,150)), (WIDTH-318,152))
        
        # Риск
        g_rect = pygame.Rect(20,180,WIDTH-40,HEIGHT-220)
        pygame.draw.rect(screen, (20,20,30), g_rect, border_radius=5)
        
        for p,col in [(10,(0,200,100)), (25,(255,170,0))]:
            y = g_rect.bottom - int((p/100)*g_rect.height)
            pygame.draw.line(screen, col, (g_rect.left,y), (g_rect.right,y), 1)
            screen.blit(font_s.render(f"{p}%", True, col), (g_rect.left+5, y-20))
        
        if len(risk_history) > 1:
            risk_list = list(risk_history)
            pts = [(g_rect.left + int(i*g_rect.width/(HISTORY_LEN-1)),
                    g_rect.bottom - int((v/100)*g_rect.height)) for i,v in enumerate(risk_list)]
            s_color = (0,200,100) if current_risk<10 else ((255,170,0) if current_risk<25 else (255,50,50))
            if len(pts) > 1:
                pygame.draw.lines(screen, s_color, False, pts, 3)
        
        screen.blit(font_l.render(f"Вероятность аварии: {current_risk:.1f}%", True, s_color),
                    (g_rect.left+20, g_rect.top+20))
        
        pygame.display.flip()
        clock.tick(FPS)

except Exception as e:
    print(f"Ошибка: {e}")
except KeyboardInterrupt:
    print("\nДетектирование остановлено пользователем.")
finally:
    # ==================== ФИНАЛ ====================
    if csv_buf:
        writer.writerows(csv_buf)
    csv_file.close()
    pygame.quit()
    
    print("\n" + "="*60)
    print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
    print("="*60)
    
    if total_pred > 0:
        avg_all = (total_all_time / total_pred) * 1000
        avg_feature = (total_feature_time / total_pred) * 1000
        avg_classifier = (total_classifier_time / total_pred) * 1000
        avg_scaler = (total_scaler_time / total_pred) * 1000
        avg_ordinal = (total_ordinal_time / total_pred) * 1000
        avg_regression = (total_regression_time / total_pred) * 1000
        
        print(f"\nAll_latency                     {avg_all:.3f} ms\n")
        print(f"feature_extraction_latency:     {avg_feature:.3f} ms")
        print(f"classification_inf_latency:     {avg_classifier:.3f} ms")
        print(f"scaler_latency:                 {avg_scaler:.3f} ms")
        print(f"ordinal_inf_latency:            {avg_ordinal:.3f} ms")
        print(f"regression_inf_latency:         {avg_regression:.3f} ms")
        
        print(f"\nПредсказаний: {total_pred}")
        print(f"Всего пакетов: {logged_pkts_total}")
        print(f"Средняя задержка: {avg_all:.4f} ms")
        print(f"Пропускная способность: {1000/avg_all:.1f} pred/sec")
        

    else:
        print("\nНет предсказаний")
    
    print("="*60)